# Librerias

In [16]:
import importlib
import subprocess
import sys

# Mapeo de imports → nombres reales de paquetes en pip
required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "seaborn": "seaborn",
    "statsmodels": "statsmodels",
    "sklearn": "scikit-learn",   
    "plotly": "plotly",
    "stargazer": "stargazer",
    "StepMix":"StepMix",
    "deep_translator":"deep-translator"
}

def safe_pip_install(package):
    """Instala un paquete si no está, sin actualizar."""
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        print(f"✓ {package} instalado correctamente.")
    except Exception:
        print(f"⚠ No se pudo instalar {package}. Puede estar bloqueado por Anaconda.")

def install_if_missing(import_name, pip_name):
    try:
        importlib.import_module(import_name)
        print(f"✔ {import_name} ya está instalado.")
    except ImportError:
        print(f"✖ {import_name} no está instalado. Instalando {pip_name}...")
        safe_pip_install(pip_name)

print("📦 Verificando librerías necesarias...\n")

for import_name, pip_name in required_packages.items():
    install_if_missing(import_name, pip_name)

print("\n🎉 Todas las librerías están listas.")


📦 Verificando librerías necesarias...

✔ numpy ya está instalado.
✔ pandas ya está instalado.
✔ matplotlib ya está instalado.
✔ scipy ya está instalado.
✔ seaborn ya está instalado.
✔ statsmodels ya está instalado.
✔ sklearn ya está instalado.
✔ plotly ya está instalado.
✔ stargazer ya está instalado.
✖ StepMix no está instalado. Instalando StepMix...
✓ StepMix instalado correctamente.
✖ deep_translator no está instalado. Instalando deep-translator...
✓ deep-translator instalado correctamente.

🎉 Todas las librerías están listas.


In [17]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
sns.set()
from stepmix.stepmix import StepMix
#from IPython.display import Image
from stepmix.utils import get_mixed_descriptor

from sklearn import preprocessing
from sklearn.cluster import KMeans
#from statsmodels.tsa.arima.model import ARIMA

from deep_translator import GoogleTranslator

import warnings
warnings.filterwarnings("ignore")

# %matplotlib inline

# Limpieza y tratamiento de datos

## Lectura de datos

In [ ]:
df_consumidores=pd.read_csv("customers.csv")
df_compras=pd.read_csv("orders.csv")

#contar nulos por columna
display(df_consumidores.isnull().sum())
display(df_compras.isnull().sum())


En general se observa que no existen valores nulos en el data de consumidores pero si existen 15796 valores nulos en el df de compras,en particular todos los nulos se encuentra en la columna customer_raiting

## Union de bases de datos

Se aprecia ademas que ambos df tiene la columna customer id,con la que se pueden unir usando una relacion 1:varios,esto ya que la columna is_repeat_customer del de compras nos señala que un cosumidor puede aparecer mas de una vez en ella

In [ ]:
df = pd.merge(
    df_consumidores,
    df_compras,
    on='customer_id',
    how='inner',
    validate='one_to_many'
)

## Traduccion columnas

ademas para mayor facilidad a la hora de intepretar las variables en los df,se traduciran las columnas al español

In [ ]:
# Diccionario de traducción al español
traduccion_columnas = {
    'customer_id': 'id_cliente',
    'country': 'pais',
    'age': 'edad',
    'gender': 'genero',
    'membership_tier': 'nivel_membresia',
    'registration_date': 'fecha_registro',
    'total_orders': 'total_pedidos',
    'total_spend_usd': 'gasto_total_usd',
    'avg_order_value_usd': 'valor_promedio_pedido_usd',
    'days_since_last_purchase': 'dias_desde_ultima_compra',
    'preferred_category': 'categoria_preferida',
    'preferred_device': 'dispositivo_preferido',
    'preferred_payment_method': 'metodo_pago_preferido',
    'acquisition_channel': 'canal_adquisicion',
    'reviews_given': 'resenas_realizadas',
    'avg_review_score': 'puntaje_promedio_resenas',
    'returns_made': 'devoluciones_realizadas',
    'wishlist_items': 'items_lista_deseos',
    'newsletter_subscribed': 'suscrito_newsletter',
    'churned': 'abandono_cliente',

    'order_id': 'id_pedido',
    'order_date': 'fecha_pedido',
    'year': 'anio',
    'month': 'mes',
    'quarter': 'trimestre',
    'day_of_week': 'dia_semana',

    'product_name': 'nombre_producto',
    'category': 'categoria',

    'unit_price_usd': 'precio_unitario_usd',
    'quantity': 'cantidad',
    'subtotal_usd': 'subtotal_usd',

    'discount_pct': 'porcentaje_descuento',
    'discount_amount_usd': 'monto_descuento_usd',

    'shipping_fee_usd': 'costo_envio_usd',

    'tax_pct': 'porcentaje_impuesto',
    'tax_amount_usd': 'monto_impuesto_usd',

    'total_amount_usd': 'monto_total_usd',

    'payment_method': 'metodo_pago',
    'device_used': 'dispositivo_usado',

    'delivery_days': 'dias_entrega',
    'delivery_date': 'fecha_entrega',

    'order_status': 'estado_pedido',
    'returned': 'devuelto',

    'customer_rating': 'calificacion_cliente',

    'session_duration_minutes': 'duracion_sesion_minutos',
    'pages_viewed_before_purchase': 'paginas_vistas_antes_compra',

    'is_repeat_customer': 'cliente_recurrente'
}

df.rename(columns=traduccion_columnas, inplace=True)

In [22]:
display(df.head(5))

,id_cliente,pais,edad,genero,nivel_membresia,fecha_registro,total_pedidos,gasto_total_usd,valor_promedio_pedido_usd,dias_desde_ultima_compra,...,metodo_pago,dispositivo_usado,dias_entrega,fecha_entrega,estado_pedido,devuelto,calificacion_cliente,duracion_sesion_minutos,paginas_vistas_antes_compra,cliente_recurrente
0,C00001,United States,40,Male,Free,2019-01-17,4,286.63,63.78,49,...,PayPal,Mobile,2,2025-09-07,Delivered,0,NaN,14.6,2,1
1,C00001,United States,40,Male,Free,2019-01-17,4,286.63,63.78,49,...,Credit Card,Mobile,2,2021-10-07,Delivered,0,NaN,3.3,15,0
2,C00002,United States,20,Female,Free,2026-03-04,11,1245.18,107.32,126,...,PayPal,Mobile,1,2025-12-25,Cancelled,0,NaN,29.9,2,1
3,C00002,United States,20,Female,Free,2026-03-04,11,1245.18,107.32,126,...,UPI / Digital Wallet,Tablet,2,2023-07-20,Delivered,0,NaN,16.8,5,1
4,C00002,United States,20,Female,Free,2026-03-04,11,1245.18,107.32,126,...,PayPal,Mobile,7,2026-04-06,Returned,1,NaN,11.4,11,0


# Seleccion variables sociodemograficas y RFM